# 🌸 Ejercicio: Clasificación de Iris con MLflow

## 🎯 Objetivos del Ejercicio

En este ejercicio práctico aprenderás a:

1. **Trabajar con un problema de clasificación** multiclase
2. **Aplicar MLflow** a un modelo de árbol de decisión
3. **Registrar experimentos** de forma profesional
4. **Evaluar y comparar** diferentes configuraciones
5. **Crear visualizaciones** para clasificación

---

## 🌺 El Dataset Iris

El **Iris Dataset** es uno de los datasets más famosos en Machine Learning, introducido por Ronald Fisher en 1936.

### 📊 Características del Dataset

| Aspecto | Descripción |
|---------|-------------|
| **Muestras** | 150 flores (50 por clase) |
| **Características** | 4 medidas físicas (cm) |
| **Clases** | 3 especies de iris |
| **Tipo** | Clasificación multiclase |
| **Dificultad** | ⭐⭐ (Principiante) |

### 🌸 Las 3 Especies de Iris

1. **Iris Setosa** (Clase 0)
2. **Iris Versicolor** (Clase 1)
3. **Iris Virginica** (Clase 2)

### 📏 Las 4 Características

1. **Sepal Length** (Longitud del sépalo)
2. **Sepal Width** (Ancho del sépalo)
3. **Petal Length** (Longitud del pétalo)
4. **Petal Width** (Ancho del pétalo)

---

## 🎯 Tu Misión

Construir un **Árbol de Decisión** que pueda clasificar correctamente las especies de iris basándose en sus medidas físicas, y documentar todo el proceso con MLflow.

---

## 🚀 ¡Empecemos!

In [0]:
import mlflow
import warnings
warnings.filterwarnings('ignore')


mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks")

# ⚠️ IMPORTANTE: Cambia esto por tu email de Databricks
email = 'pelayosierra@protonmail.com'  # Ejemplo: 'nombre.apellido@ejemplo.com'

# Validar que el email no esté vacío
if not email:
    print("⚠️  ADVERTENCIA: Debes configurar tu email antes de continuar")
    print("   Cambia la variable 'email' por tu email de Databricks")
else:
    # Configurar el experimento
    experiment_name = f"/Users/{email}/4-ejercicio-the-irish"
    mlflow.set_experiment(experiment_name)
    
    print("=" * 60)
    print("✅ MLflow configurado correctamente")
    print("=" * 60)
    print(f"📊 Experimento: {experiment_name}")
    print(f"🌸 Dataset: Iris (Clasificación)")
    print(f"🤖 Modelo: Decision Tree Classifier")
    print("=" * 60)

## ⚙️ Paso 1: Configuración de MLflow

Configuramos el experimento donde se registrarán todas las ejecuciones.

**🔴 IMPORTANTE**: Cambia el email vacío por tu email de Databricks.

## 📚 Paso 2: Importar Librerías

Importamos todas las herramientas necesarias para clasificación.

In [0]:
%pip install seaborn

# Librerías principales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# MLflow para tracking
import mlflow
import mlflow.sklearn

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn import datasets
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix,
    classification_report
)

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")

print("✅ Todas las librerías importadas correctamente")
print("   - MLflow: Listo para tracking")
print("   - Scikit-learn: Modelos y métricas")
print("   - Matplotlib & Seaborn: Visualizaciones")

## 🌺 Paso 3: Cargar y Explorar el Dataset Iris

Cargaremos el famoso dataset de flores Iris y exploraremos sus características.

In [0]:
# TODO: Carga el dataset Iris y explora su estructura.
# Pistas:
# - Usa datasets.load_iris()
# - Crea un DataFrame con dataset.data y dataset.feature_names
# - Añade la columna target/especie
# - Muestra shape, clases, primeras filas y estadísticas descriptivas

dataset = datasets.load_iris()
df = pd.DataFrame(dataset.data, columns=dataset.feature_names)
df['species'] = dataset.target
print(f"Shape:\n{df.shape}")
print(f"\nHead:\n{df.head()}")
print(f"\nDescribe:\n{df.describe()}")


### 📊 Visualización Exploratoria del Dataset

Visualicemos las relaciones entre las características para entender mejor los datos.

In [0]:
# -- AVISO -- Esta sección ha sido parcialmente resuelta con el uso de IA generativa
# Mapeo de especies para leyendas legibles
species_map = {0: 'Setosa', 1: 'Versicolor', 2: 'Virginica'}
df['species_name'] = df['species'].map(species_map)

fig, axes = plt.subplots(3, 2, figsize=(15, 18))
fig.suptitle('Exploración del Dataset Iris', fontsize=16, fontweight='bold')

# 1. Scatter: longitud vs ancho del pétalo por especie
ax = axes[0, 0]
for spec, color in zip(['Setosa', 'Versicolor', 'Virginica'], ['#e74c3c', '#2ecc71', '#3498db']):
    subset = df[df['species_name'] == spec]
    ax.scatter(subset['petal length (cm)'], subset['petal width (cm)'],
               label=spec, alpha=0.7, s=60, c=color, edgecolors='white')
ax.set_xlabel('Longitud del pétalo (cm)')
ax.set_ylabel('Ancho del pétalo (cm)')
ax.set_title('Pétalo: Longitud vs Ancho')
ax.legend()

# 2. Scatter: longitud vs ancho del sépalo por especie
ax = axes[0, 1]
for spec, color in zip(['Setosa', 'Versicolor', 'Virginica'], ['#e74c3c', '#2ecc71', '#3498db']):
    subset = df[df['species_name'] == spec]
    ax.scatter(subset['sepal length (cm)'], subset['sepal width (cm)'],
               label=spec, alpha=0.7, s=60, c=color, edgecolors='white')
ax.set_xlabel('Longitud del sépalo (cm)')
ax.set_ylabel('Ancho del sépalo (cm)')
ax.set_title('Sépalo: Longitud vs Ancho')
ax.legend()

# 3. Boxplots de longitud del pétalo por especie
ax = axes[1, 0]
sns.boxplot(data=df, x='species_name', y='petal length (cm)', ax=ax, palette='Set2')
ax.set_title('Distribución: Longitud del Pétalo por Especie')
ax.set_xlabel('Especie')

# 4. Histograma de longitud del pétalo por especie
ax = axes[1, 1]
for spec, color in zip(['Setosa', 'Versicolor', 'Virginica'], ['#e74c3c', '#2ecc71', '#3498db']):
    subset = df[df['species_name'] == spec]
    ax.hist(subset['petal length (cm)'], bins=12, alpha=0.5, label=spec, color=color)
ax.set_xlabel('Longitud del pétalo (cm)')
ax.set_ylabel('Frecuencia')
ax.set_title('Histograma: Longitud del Pétalo')
ax.legend()

# 5. Histograma de longitud del sépalo por especie
ax = axes[2, 0]
for spec, color in zip(['Setosa', 'Versicolor', 'Virginica'], ['#e74c3c', '#2ecc71', '#3498db']):
    subset = df[df['species_name'] == spec]
    ax.hist(subset['sepal length (cm)'], bins=12, alpha=0.5, label=spec, color=color)
ax.set_xlabel('Longitud del sépalo (cm)')
ax.set_ylabel('Frecuencia')
ax.set_title('Histograma: Longitud del Sépalo')
ax.legend()

# 6. Matriz de correlación
ax = axes[2, 1]
corr = df.drop(columns=['species', 'species_name']).corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax,
            xticklabels=['Sep.L', 'Sep.W', 'Pet.L', 'Pet.W'],
            yticklabels=['Sep.L', 'Sep.W', 'Pet.L', 'Pet.W'])
ax.set_title('Matriz de Correlación')

plt.tight_layout()
plt.show()

print("\nObservaciones clave:")
print("   • PÉTALO: Setosa es claramente separable (pétalos cortos y estrechos <2cm). Versicolor y Virginica se distinguen bastante bien.")
print("   • SÉPALO: Mayor solapamiento entre especies, especialmente Versicolor y Virginica. Setosa tiende a tener sépalos más anchos pero cortos.")
print("   • Comparación: las medidas del pétalo separan mejor las especies que las del sépalo.")
print("   • Alta correlación entre longitud y ancho del pétalo (características redundantes).")
print("   • El sépalo tiene baja correlación entre sus dimensiones y menor poder discriminativo.")


## 🔀 Paso 4: Preparar los Datos

Dividimos el dataset en conjuntos de entrenamiento y prueba.

In [0]:
# Separa características (X) y etiquetas (y).
X = df.drop(columns=['species', 'species_name'])
y = df['species']

# Divide el dataset en entrenamiento y prueba.
# Pistas:
# - Usa train_test_split
# - Reserva una parte para test
# - Usa random_state para reproducibilidad
# - Considera stratify=y para mantener la proporción de clases

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y)

# Comprueba tamaños y distribución de clases en cada conjunto
print(f"Tamaño X_train: {X_train.shape}, Tamaño X_test: {X_test.shape}")
print(f"Tamaño y_train: {y_train.shape}, Tamaño y_test: {y_test.shape}")
print(f"\nDistribución de clases - Train:\n{y_train.value_counts().sort_index()}")
print(f"\nDistribución de clases - Test:\n{y_test.value_counts().sort_index()}")
print(f"\nProporción Train (%):\n{(y_train.value_counts(normalize=True).sort_index() * 100).round(1)}")
print(f"\nProporción Test (%):\n{(y_test.value_counts(normalize=True).sort_index() * 100).round(1)}")




## 🌳 Paso 5: Entrenar Árbol de Decisión con MLflow

Ahora entrenaremos un modelo de **Decision Tree** y registraremos todo con MLflow.

### 🔑 Hiperparámetros del Árbol de Decisión

- **max_depth**: Profundidad máxima del árbol (evita overfitting)
- **max_features**: Número máximo de características a considerar por split
- Valores más altos = modelo más complejo = mayor riesgo de overfitting

In [0]:
# ========================================
# TODO: ENTRENAMIENTO CON MLFLOW
# ========================================

# TODO: Activa autologging de MLflow.
mlflow.sklearn.autolog()

# TODO: Inicia un run con un nombre descriptivo.
with mlflow.start_run(run_name="Decision Tree - Iris Classifier") as run:
# Define hiperparámetros del árbol.
    max_depth = 3 # es un dataset pequeño, no requiere mucha profundidad
    max_features = None # usamos las 4
    random_state = 42
#
#   Crea y entrena DecisionTreeClassifier.
    dt = DecisionTreeClassifier(
        max_depth=max_depth,
        max_features=max_features,
        random_state=random_state
    )
    dt.fit(X_train, y_train)
#
#   Genera predicciones para train y test.
    y_pred_train = dt.predict(X_train)
    y_pred_test = dt.predict(X_test)
#
#   Calcula métricas de clasificación.
    accuracy_test = accuracy_score(y_test, y_pred_test)
    precision_test = precision_score(y_test, y_pred_test, average="micro")
    recall_test = recall_score(y_test, y_pred_test, average="micro")
    f1_test = 2 * (precision_test * recall_test) / (precision_test + recall_test)

    classification_metrics = {
        "accuracy_test": accuracy_test,
        "precision_test": precision_test,
        "recall_test": recall_test,
        "f1_test": f1_test
    }

    for m, val in classification_metrics.items():
        mlflow.log_metric(m, val)
#
#   Evalúa con cross-validation.
    cv_scores = cross_val_score(dt, X_train, y_train, cv=5, scoring='accuracy')
    mlflow.log_metric("cv_accuracy", cv_scores.mean())  
    mlflow.log_metric("cv_std", cv_scores.std())
    print(cv_scores)
    print(f"CV Accuracy: {cv_scores.mean():.4f}")
    print(f"CV Std: {cv_scores.std():.4f}")
#
#   Visualiza matriz de confusión y, opcionalmente, el árbol.
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    import matplotlib.pyplot as plt

    # Calcular matriz
    titles_options = [
        ("Confusion matrix, without normalization", None),
        ("Normalized confusion matrix", "true"),
    ]
    for title, normalize in titles_options:
        disp = ConfusionMatrixDisplay.from_estimator(
            dt,
            X_test,
            y_test,
            display_labels=["Setosa", "Versicolor", "Virginica"],
            cmap=plt.cm.Blues,
            normalize=normalize
        )
        disp.ax_.set_title(title)

        print(title)
        print(disp.confusion_matrix)

    plt.show()

    # Matriz TP / TN / FP / FN por clase (one-vs-rest)
    cm = confusion_matrix(y_test, y_pred_test)
    classes = ["Setosa", "Versicolor", "Virginica"]
    matrix_data = []
    for i in range(len(classes)):
        TP = cm[i, i]
        FP = cm[:, i].sum() - TP
        FN = cm[i, :].sum() - TP
        TN = cm.sum() - TP - FP - FN
        matrix_data.append([TP, TN, FP, FN])

    tp_tn_fp_fn_df = pd.DataFrame(
        matrix_data,
        index=classes,
        columns=['TP', 'TN', 'FP', 'FN']
    )

    fig, ax = plt.subplots(figsize=(8, 4))
    sns.heatmap(tp_tn_fp_fn_df, annot=True, fmt='d', cmap='YlGnBu', ax=ax,
                linewidths=0.5, linecolor='gray')
    ax.set_title('TP / TN / FP / FN por clase (One-vs-Rest)')
    plt.tight_layout()
    plt.show()

    print("\nTP / TN / FP / FN por clase:")
    print(tp_tn_fp_fn_df)

#     TODO: Escribe una breve interpretación de los resultados.
# Los resultados demuestran que el modelo de árbol de decisión es capaz de clasificar
# correctamente la mayoría de las muestras de Iris, con una precisión del 96.7% en 
# el conjunto de prueba. Además, el modelo muestra una alta sensibilidad (recall) y 
# especificidad (precision) en todas las clases, lo que indica que es un modelo robusto 
# y generalizable. La matriz de confusión muestra que el modelo cometió un error en la 
# clasificación de una sola muestra de la clase 'Virginica'.

## 🎯 Reflexión del Ejercicio

Completa esta sección cuando termines el notebook.

### ✅ Comprueba que has trabajado

1. [ ] Exploración del dataset Iris
2. [ ] Separación train/test
3. [ ] Entrenamiento de un Árbol de Decisión
4. [ ] Evaluación con métricas de clasificación
5. [ ] Registro del experimento en MLflow
6. [ ] Interpretación de resultados

---

## 📚 Conceptos Clave - Clasificación

### 🎯 Métricas de Clasificación

| Métrica | Descripción | Cuándo Usarla |
|---------|-------------|---------------|
| **Accuracy** | % de predicciones correctas | Clases balanceadas |
| **Precision** | % de positivos correctos | Importante evitar falsos positivos |
| **Recall** | % de positivos encontrados | Importante encontrar todos los positivos |
| **F1-Score** | Media armónica de P y R | Balance entre precisión y recall |

### 🌳 Árbol de Decisión

**TODO: Explica con tus palabras:**
- ¿Qué ventajas tiene un árbol de decisión?
- ¿Qué riesgos tiene respecto a overfitting?
- ¿Cómo interpretarías una matriz de confusión multiclase?

---

## 🚀 Desafíos Adicionales

### 🎯 Desafío 1: Optimiza los Hiperparámetros

Prueba diferentes configuraciones y encuentra la mejor:

```python
# Experimenta con:
max_depth = [3, 5, 10, None]
max_features = [2, 3, 4, 'sqrt']
min_samples_split = [2, 5, 10]
```

**TODO:** ¿Qué combinación da el mejor balance entre accuracy y overfitting?

### 🎯 Desafío 2: Implementa Grid Search

Automatiza la búsqueda del mejor modelo con `GridSearchCV`.

### 🎯 Desafío 3: Compara con Otros Modelos

Entrena y compara Random Forest, SVM, KNN o Logistic Regression.

### 🎯 Desafío 4: Análisis de Errores

**TODO:** ¿Qué flores se confunden más y por qué?

### 🎯 Desafío 5: Reducción de Dimensionalidad

Usa PCA para reducir a 2 dimensiones y visualizar límites de decisión.


In [0]:
# 🎨 DESAFÍOS OPCIONALES
# Usa este espacio para completar los desafíos del ejercicio.

# ========================================
# TODO: DESAFÍO 1 - Optimización manual
# ========================================
# configuraciones = [...]
# for config in configuraciones:
#     # Entrena, evalúa y registra cada configuración en MLflow
#     pass

# ========================================
# TODO: DESAFÍO 2 - Grid Search
# ========================================
# from sklearn.model_selection import GridSearchCV
# param_grid = {...}
# grid_search = GridSearchCV(...)
# grid_search.fit(...)

# ========================================
# TODO: DESAFÍO 3 - Comparación de modelos
# ========================================
# modelos = {...}
# resultados = []
# for nombre, modelo in modelos.items():
#     # Entrena y compara métricas
#     pass

# ========================================
# TODO: DESAFÍO 4 - Análisis de errores
# ========================================
# incorrect_indices = ...
# TODO: inspecciona las muestras mal clasificadas.

# ========================================
# TODO: DESAFÍO 5 - PCA y visualización 2D
# ========================================
# from sklearn.decomposition import PCA
# pca = PCA(n_components=2)
# X_pca = ...
